## Transform Order Data  - Explode Arrays

1. Access elements from the JSON object
2. Deduplicate Array Elements
3. Explode Arrays
4. Write the Transformed Data to Silver Schema

In [0]:
df_orders = spark.table('gizmobox.silver.py_orders_json')
display(df_orders)

1. Access elements from the JSON object

In [0]:
from pyspark.sql import functions as F

In [0]:
df_orders_normalized = (
    df_orders
    .select (
       "json_value.order_id",
       "json_value.order_status",
       "json_value.payment_method",
       "json_value.total_amount",
       "json_value.transaction_timestamp",
       "json_value.customer_id",
       "json_value.items"
    )
)
display(df_orders_normalized)

2. Deduplicate Array Elements

In [0]:
df_orders_normalized = (
    df_orders
    .select (
       "json_value.order_id",
       "json_value.order_status",
       "json_value.payment_method",
       "json_value.total_amount",
       "json_value.transaction_timestamp",
       "json_value.customer_id",
       F.array_distinct("json_value.items").alias("items")
    )
)
display(df_orders_normalized)

3. Explode Arrays

In [0]:
df_orders_exploded = (
    df_orders_normalized
    .select (
       "order_id",
       "order_status",
       "payment_method",
       "total_amount",
       "transaction_timestamp",
       "customer_id",
       F.explode("items").alias("item")
    )
)
display(df_orders_exploded)

In [0]:
df_order_items = (
    df_orders_exploded
    .select (
       "order_id",
       "order_status",
       "payment_method",
       "total_amount",
       "transaction_timestamp",
       "customer_id",
       "item.item_id",
       "item.name",
       "item.price",
       "item.quantity",
       "item.category",
       "item.details.brand",
       "item.details.color"
    )
)
display(df_order_items)#not actually required here

4. Write the Transformed Data to Silver Schema

In [0]:
df_order_items.writeTo("gizmobox.silver.py_orders").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders    